# RAG 마스터 #1 - Practice (맥/LM Studio 교정판)

- 자료의 `unsloth/gemma-4-e2b-it` -> **`gemma-4-e2b-it`** 로 교정 (LM Studio 실제 모델 ID)
- base_url: `http://host.docker.internal:12345/v1` (컨테이너 -> 맥 네이티브 LM Studio)
- 라이브러리는 이미지에 미리 설치됨 -> `!pip install` 불필요
- 전제: LM Studio 서버 Running(12345) + gemma 모델 Load 상태

## Practice 1-1. 랭체인 없이 호출하기

In [1]:
from openai import OpenAI

client = OpenAI(  # LM Studio 로컬 서버에 연결
    base_url="http://host.docker.internal:12345/v1",
    api_key="lm-studio",  # 임의의 문자열(검증 안 함)
)

response = client.chat.completions.create(  # "안녕하세요!" 메시지를 보내고 응답을 받음
    model="gemma-4-e2b-it",  # LM Studio에 로드된 모델명 (교정)
    messages=[{"role": "user", "content": "안녕하세요!"}],
)
print(response.choices[0].message.content)

안녕하세요! 저는 Gemini라고 합니다. 무엇을 도와드릴까요? 😊


In [2]:
from typing import List

prompt_template = "주제 {topic}에 대해 짧은 설명을 해주세요."  # 프롬프트 템플릿

def call_chat_model(messages: List[dict]):  # 메시지를 보내고 응답을 받는 함수
    response = client.chat.completions.create(
        model="gemma-4-e2b-it",
        messages=messages,
    )
    return response.choices[0].message.content

def invoke_chain(topic: str):  # 주어진 주제로 설명을 요청하는 함수
    prompt_value = prompt_template.format(topic=topic)
    messages = [{"role": "user", "content": prompt_value}]
    return call_chat_model(messages)

print(invoke_chain("더블딥"))

## 더블딥(Double Dip)에 대한 짧은 설명

**더블딥(Double Dip)**은 **한 가지 행동이나 활동을 두 번 반복하는 행위**를 의미합니다.

쉽게 말해, **"한 번 한 것을 다시 한번 하는 것"**이라고 생각하시면 됩니다.

**주요 특징:**

* **반복성:** 동일하거나 매우 유사한 행동이 연속적으로 발생합니다.
* **의미:** 상황이나 맥락에 따라 긍정적일 수도 있고(예: 좋은 기회를 두 번 잡음), 부정적일 수도 있습니다(예: 같은 실수를 반복함).

**예시:**

* **비즈니스:** 동일한 고객에게 두 번 연속으로 똑같은 제안을 하는 것.
* **일상생활:** 이미 끝난 일을 다시 시작하거나, 같은 음식을 두 번 연속으로 먹는 것.

**요약하자면, '더블딥'은 어떤 행위를 중복해서 수행하는 것을 나타내는 용어입니다.**


## Practice 1-2. 랭체인으로 호출하기 (LCEL 파이프라인)

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

prompt = ChatPromptTemplate.from_template("주제 {topic}에 대해 짧은 설명을 해주세요.")

model = ChatOpenAI(  # LM Studio 로컬 서버의 모델 사용
    model="gemma-4-e2b-it",  # 교정
    base_url="http://host.docker.internal:12345/v1",
    api_key="lm-studio",
    temperature=0.7,
)
output_parser = StrOutputParser()

# 파이프라인: 주제 -> 프롬프트 -> 모델 -> 문자열 파싱
chain = (
    {"topic": RunnablePassthrough()}
    | prompt
    | model
    | output_parser
)

print(chain.invoke("더블딥"))

## 주제 더블딥(Topic Double-Dip) 설명

**주제 더블딥(Topic Double-Dip)**은 **하나의 아이디어나 주제를 두 번 이상, 혹은 여러 각도에서 반복하거나 깊이 있게 다루는 전략 또는 현상**을 의미합니다. 단순히 같은 내용을 반복하는 것이 아니라, **다양한 관점, 맥락, 또는 심층적인 분석을 통해 동일한 핵심 주제에 대해 반복적으로 접근하여 이해도를 높이거나 메시지를 강화하는 방식**이라고 볼 수 있습니다.

### 주요 특징:

1. **반복과 심화:** 동일한 주제를 다른 각도(예: 기술적 측면, 사회적 영향, 개인적 경험 등)에서 다시 다루어 주제에 대한 이해의 깊이를 더합니다.
2. **다각적 접근:** 한 가지 관점에만 머무르지 않고, 여러 층위(Layer)를 통해 주제를 조명하여 입체적인 시각을 제공합니다.
3. **강조 효과:** 반복을 통해 핵심 메시지를 청중이나 독자에게 확실하게 각인시키는 효과가 있습니다.
4. **맥락 전환:** 각 반복마다 다른 맥락(Context)을 부여함으로써 주제의 의미가 확장될 수 있습니다.

### 사용되는 분야:

* **콘텐츠 제작 (글쓰기, 영상):** 하나의 이슈를 여러 콘텐츠 형식(블로그 글, 인터뷰, 다큐멘터리 등)으로 깊이 있게 다룰 때 사용됩니다.
* **학술 연구:** 특정 이론이나 현상을 여러 실험이나 분석 틀을 통해 검증할 때 활용될 수 있습니다.
* **마케팅/커뮤니케이션:** 핵심 가치(Value Proposition)를 다양한 스토리텔링 방식으로 반복하여 잠재 고객에게 각인시킬 때 사용됩니다.

**요약하자면, 주제 더블딥은 '같은 것을 여러 번 다루되, 매번 다른 깊이와 관점을 부여하여 주제의 통찰력을 극대화하는 전략'입니다.**
